In [1]:
import warnings
warnings.filterwarnings('ignore')
import logging
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help
import QuantStudio.api as QS
QS.Core.setDefaultLogLevel(level=logging.WARNING)

基于 SQL 数据库构建的因子库的对应关系：
* 整个数据库对应于因子库
* 每张数据库表对应于因子表
* 每张数据库表的字段对应于单个因子

本质上是将一个二维的因子数据矩阵挤压成具有二重索引的一维向量. 对于因子数据的访问, 内部使用标准的 SQL 查询语句完成. 

![SQL因子库](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/SQL因子库.png)

# 外部因子库

## 聚源因子库

以下代码要求有可以访问的聚源数据库，且设置好了配置文件。或者执行 [import_postgres_jydb_demo_data.sh](../tools/import_postgres_jydb_demo_data.sh) 脚本生成示例数据，这要求有写入权限的 postgresql 数据库。

聚源因子库的配置文件默认位于用户目录下的 “QuantStudioConfig” 文件夹里的 "JYDBConfig.json" 文件。通常将数据库的连接信息配置到文件里，示例如下:
```json
{
    "Name": "JYDB",
    "DBType": "PostgreSQL",
    "DBName": "JYDB",
    "IPAddr": "localhost",
    "Port": 5432,
    "User": "postgres",
    "Pwd": "123456",
    "TablePrefix": "",
    "CharSet": "utf8",
    "Connector": "default"
}
```

In [2]:
# 创建聚源因子库
from QuantStudio.Factor.JYDB import JYDB

FDB = JYDB().connect()
print(qs_help(FDB))

类型: JYDB
模块: QuantStudio.Factor.JYDB
QS 对象类型: 因子库
QS 对象名称: JYDB
QSID: a4602c3052c0032aa821b2f2c4db8e989d51e33a046ece1e6b39f92f1d34c0a2
参数集:
    * Name(名称): <class 'str'>, 默认值 'JYDB', 当前取值: 'JYDB'
    * DBType(数据库类型): typing.Literal['MySQL', 'SQL Server', 'Oracle', 'PostgreSQL'], 默认值 'MySQL', 当前取值: 'PostgreSQL'
    * DBName(数据库名): <class 'str'>, 默认值 'Scorpion', 当前取值: 'JYDB'
    * IPAddr(IP地址): <class 'str'>, 默认值 '127.0.0.1', 当前取值: '127.0.0.1'
    * Port(端口): <class 'int'>, 默认值 3306, 当前取值: 5433
    * User(用户名): <class 'str'>, 默认值 'root', 当前取值: 'shzq'
    * Pwd(密码): <class 'str'>, 默认值 '', 当前取值: 'shzq#321'
    * TablePrefix(表名前缀): <class 'str'>, 默认值 '', 当前取值: ''
    * CharSet(字符集): typing.Literal['utf8', 'utf8mb4', 'gbk', 'gb2312', 'gb18030', 'cp936', 'big5'], 默认值 'utf8', 当前取值: 'utf8'
    * Connector(连接器): typing.Literal['default', 'cx_Oracle', 'pymssql', 'mysql.connector', 'pymysql', 'psycopg2', 'pyodbc'], 默认值 'default', 当前取值: 'default'
    * ConnRetryNum(连接重试次数): <class 'int'>, 默认值 3, 当前

### 获取时点序列

#### 获取交易日序列

In [5]:
print(qs_help(FDB.getTradeDay))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getTradeDay(start_date: Optional[datetime.datetime] = None, end_date: Optional[datetime.datetime] = None, exchange: Literal['SSE', 'SZSE', 'SHFE', 'DCE', 'CZCE', 'INE', 'CFFEX'] = 'SSE', **kwargs) -> List[datetime.datetime]
说明文档:
    给定交易所、起始日和结束日, 获取交易日序列
    
    Args:
        start_date: 起始日, None 表示从 1900-01-01 开始
        end_date: 结束日, None 表示当前日期
        exchange: 交易所, 默认 SSE(上交所)
    
    Returns:
        交易日序列


In [6]:
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 8), exchange="SSE")
print(DTs[:5])

[datetime.datetime(2022, 1, 4, 0, 0), datetime.datetime(2022, 1, 5, 0, 0), datetime.datetime(2022, 1, 6, 0, 0), datetime.datetime(2022, 1, 7, 0, 0)]


### 获取ID序列

#### 股票 ID

In [8]:
print(qs_help(FDB.getStockID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getStockID(exchange: Union[Literal['SSE', 'SZSE', 'BSE', 'HKEX', 'AMEX', 'NASDAQ', 'NYSE', 'NEEQ'], Tuple[Literal['SSE', 'SZSE', 'BSE', 'HKEX', 'AMEX', 'NASDAQ', 'NYSE', 'NEEQ']]] = ('SSE', 'SZSE', 'BSE'), date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定交易所和日期, 获取股票证券 ID 序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 默认 ("SSE", "SZSE", "BSE") 表示上交所、深交所、北交所
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日期在指定日之前的股票, True 表示上市日期在指定日之前且尚未退市的股票
        start_date: 起始日, 如果非 None 并且 is_current=False 表示提取在 start_date 至 date 之间上市过的股票, 如果 is_current=True 表示提取在 start_date 至 date 之间均保持上市的股票
    
    Returns:
        股票证券 ID 序列


In [9]:
IDs = FDB.getStockID(exchange=("SSE", "SZSE", "BSE"), date=dt.datetime(2022, 1, 1), is_current=True, start_date=None)
print(IDs[:5])

['000001.SZ', '000002.SZ', '000004.SZ', '000005.SZ', '000006.SZ']


#### 公募基金 ID

In [10]:
print(qs_help(FDB.getMutualFundID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getMutualFundID(exchange: Union[str, Tuple[str], NoneType] = None, date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定日期, 获取公募基金 ID 序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 如果非 None 表示只考虑在这些指定的交易所上市的基金, None 表示包括非上市基金
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示成立日在指定日之前的基金, True 表示成立日在指定日之前且尚未清盘的基金
    
    Returns:
        公募基金证券 ID 序列


In [ ]:
IDs = FDB.getMutualFundID(date=dt.datetime(2022, 1, 1), is_current=True, start_date=None)
print(IDs[:5])

#### 债券 ID

In [11]:
print(qs_help(FDB.getBondID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getBondID(exchange: Union[str, Tuple[str], NoneType] = None, date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定交易所和日期, 获取债券证券 ID 序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 默认 None 表示所有交易所
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示存续起始日在指定日之前的债券, True 表示存续起始日在指定日之前且尚未到期的债券
        start_date: 起始日, 如果非 None 并且 is_current=False 表示提取在 start_date 至 date 之间存续过的债券, 如果 is_current=True 表示提取在 start_date 至 date 之间均保持存续的债券
    
    Returns:
        债券证券 ID 序列


In [ ]:
IDs = FDB.getBondID(exchange=None, date=dt.datetime(2022, 1, 1), is_current=True, start_date=None)
print(IDs[:5])

#### 期货 ID

In [12]:
print(qs_help(FDB.getFutureID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getFutureID(exchange: Union[str, Tuple[str], NoneType] = None, future_code: Union[str, Tuple[str], NoneType] = None, date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定交易所、期货品种代码和日期, 获取期货证券 ID 序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 默认 None 表示所有交易所
        future_code: 期货品种代码(str)或者期货品种代码列表(tuple), None 表示所有期货品种代码
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日在指定日之前的期货, True 表示上市日在指定日之前且尚未退市的期货
        start_date: 起始日, 如果非 None 并且 is_current=False 表示提取在 start_date 至 date 之间上市过的期货, 如果 is_current=True 表示提取在 start_date 至 date 之间均保持上市的期货
        kwargs:
            contract_type: 合约类型, 可选 "月合约", "连续合约", "所有", 默认值 "月合约"
            continue_contract_type: 连续合约类型, list[str], 可选 "主力合约", "期货指数", "次主力合约", "连续合约", "连一合约", "连二合约", "连三合约", "连四合约", "当月连续合约", "次月连续合约", "当季连续合约", "下季连续合约", "隔季连续合约", 默

In [13]:
IDs = FDB.getFutureID(exchange="CFFEX", future_code="IF", date=dt.datetime(2022, 1, 1), is_current=True, start_date=None, contract_type="月合约")
print(IDs[:5])

['IF2201.CFE', 'IF2202.CFE', 'IF2203.CFE', 'IF2206.CFE']


#### 期货品种

In [14]:
print(qs_help(FDB.getFutureCode))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getFutureCode(exchange: Union[str, Tuple[str], NoneType] = None, date: Optional[datetime.datetime] = None, is_current: bool = True, **kwargs) -> List[str]
说明文档:
    给定交易所和日期, 获取期货品种代码序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 默认 None 表示所有交易所
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日在指定日之前的期货品种, True 表示上市日在指定日之前且尚未退市的期货品种
    
    Returns:
        期货品种代码序列


In [16]:
IDs = FDB.getFutureCode(exchange="CFFEX", date=None, is_current=True)
print(IDs)

['IC', 'IF', 'IH', 'IM', 'IZ', 'T', 'TF', 'TL', 'TS']


#### 期权 ID

In [17]:
print(qs_help(FDB.getOptionID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getOptionID(option_code: str = '510050', date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定期权品种代码和日期, 获取期权证券 ID 序列
    
    Args:
        option_code: 期权品种代码
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日在指定日之前的期权, True 表示上市日在指定日之前且尚未退市的期权
    
    Returns:
        期权证券 ID 序列


In [18]:
IDs = FDB.getOptionID(option_code="510050", date=None, is_current=True, start_date=None)
print(IDs[:5])

['510050C2603A02700', '510050C2603A02750', '510050C2603A02800', '510050C2603A02850', '510050C2603A02900']


#### 行业 ID

In [19]:
print(qs_help(FDB.getIndustryID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getIndustryID(standard: str = '中信行业分类', level: int = 1, date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定行业分类和日期, 获取行业 ID 序列
    
    Args:
        standard: 行业分类
        level: 分类层级
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示指定日之前曾经存续过的行业, True 表示指定日仍然保持存续的行业
        start_date: 起始日, 如果非 None 并且 is_current=False 表示提取在 start_date 至 date 之间存续过的行业, 如果 is_current=True 表示提取在 start_date 至 date 之间均保持存续的行业
    
    Returns:
        行业 ID 序列


In [20]:
IDs = FDB.getIndustryID(standard="中信行业分类", level=1, date=None, is_current=True, start_date=None)
print(IDs[:5])

['10', '11', '12', '20', '21']


# 因子表

In [3]:
# 因子表列表
print(FDB.TableNames[:5])

['交易日表(新)', '融资融券交易总量', '沪(深)港通交易日', '沪港通额度信息', '深港通额度信息']


In [4]:
# 因子列表
FT = FDB.getTable("日行情表")
print(FT.FactorNames)

['证券内部编码', '交易日', '昨收盘(元)', '今开盘(元)', '最高价(元)', '最低价(元)', '收盘价(元)', '成交量(股)', '成交金额(元)', '成交笔数(笔)']


In [5]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]

FT = FDB.getTable("日行情表", args={"LookBack": 0})
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

print("-" * 10)
print("固定 ID 的切片数据: ", Data.iloc[:, :, 0], sep="\n")

因子表数据
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 1 (minor_axis)
Items axis: 收盘价(元) to 今开盘(元)
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000001.SZ
----------
固定 ID 的切片数据: 
            收盘价(元)  今开盘(元)
2025-01-01     NaN     NaN
2025-01-02   11.43   11.73
2025-01-03   11.38   11.44
2025-01-04     NaN     NaN
2025-01-05     NaN     NaN


# 因子表类型

## WideTable

数据库数据示例：

![Wide_Table_Demo](../images/Wide_Table_Demo.png)

In [6]:
FT = FDB.getTable("日行情表")
print(qs_help(FT))

类型: _WideTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 日行情表
QSID: a56c7bd51fb7de8ea1c14d36ca61e71ea5f8af32699e8eb687568345841ed354
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '日行情表'
    * FilterCondition(筛选条件): <class 'str'>, 默认值 '', 形成 SQL 查询时附加到 WHERE 子句上的条件. 比如 "({Table}.field1>10) AND ({Table}.field2 IN ('a','b')", 其中 {Table} 会自动替换为相应的数据库表名, 当前取值: ''
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'WideTable'
    * PreFilterID(预筛选ID): <class 'bool'>, 默认值 True, 是否在 SQL 查询中筛选 ID, 如果为 True, 则在形成的 SQL 查询中的 WHERE 子句中会有 {Table}.ID字段 IN (...) 条件, 否则为 {Table}.ID字段 IS NOT NULL. 如果提取数据的 ID 不多，建议为 True, 当前取值: True
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '交易日'
    * IDField(ID字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示 ID 维度的字段名, 当前取值: None
    * DTFmt(时点格式): <class 'str'>, 默认值 '', 当前取值: ''
    * DateFmt(日期格式): <class 'str'>, 默认值 '', 当前取值

### 缺失填充

In [ ]:
# WideTable 不填充缺失, LookBack = 0
FT = FDB.getTable("日行情表", args={
    "LookBack": 0,
    "OnlyStartLookBack": False,
    "OnlyLookBackNontarget": False,
    "OnlyLookBackDT": False
})

DTs = [dt.datetime(2023, 12, 29) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

            收盘价(元)  今开盘(元)
2023-12-29    9.39    9.42
2023-12-30     NaN     NaN
2023-12-31     NaN     NaN
2024-01-01     NaN     NaN
2024-01-02    9.21    9.39


In [14]:
# WideTable 填充缺失, LookBack > 0
FT = FDB.getTable("日行情表", args={
    "LookBack": 2,
    "OnlyStartLookBack": False,
    "OnlyLookBackNontarget": False,
    "OnlyLookBackDT": False
})

DTs = [dt.datetime(2023, 12, 29) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

            收盘价(元)  今开盘(元)
2023-12-29    9.39    9.42
2023-12-30    9.39    9.42
2023-12-31    9.39    9.42
2024-01-01     NaN     NaN
2024-01-02    9.21    9.39


In [16]:
# WideTable 填充缺失, LookBack > 0, OnlyStartLookBack=True
FT = FDB.getTable("日行情表", args={
    "LookBack": 2,
    "OnlyStartLookBack": True,
    "OnlyLookBackNontarget": False,
    "OnlyLookBackDT": False
})

DTs = [dt.datetime(2023, 12, 30) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

            收盘价(元)  今开盘(元)
2023-12-30    9.39    9.42
2023-12-31     NaN     NaN
2024-01-01     NaN     NaN
2024-01-02    9.21    9.39
2024-01-03    9.20    9.19


### 公告时点

数据库数据示例：

![Wide_Table_AnnDT_Demo](../images/Wide_Table_AnnDT_Demo.png)

In [5]:
DTs = [dt.datetime(2023, 6, 30), dt.datetime(2023, 9, 30), dt.datetime(2023, 10, 1)]
IDs = ["000001.SZ"]

# 原始数据
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": 0,
    "DTField": "截止日期",
    "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1", "是否调整": "2"}
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

# 不考虑公告时点, 下面例子中 2023-10-01 使用了 2023-09-30 的数据进行了填充
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": np.inf,
    "DTField": "截止日期",
    "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1", "是否调整": "2"}
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("-" * 10)
print(Data)

# 考虑公告时点, 下面例子中 2023-10-01 使用了 2023-06-30 的数据进行了填充
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": np.inf,
    "DTField": "截止日期",
    "PublDTField": "信息发布日期",
    "AdditionalCondition": {"是否合并": "1", "是否调整": "2"}
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("-" * 10)
print(Data)

               信息发布日期             资产总计
2023-06-30 2023-08-24  5500524000000.0
2023-09-30 2023-10-25  5516388000000.0
2023-10-01        NaT              NaN
----------
               信息发布日期             资产总计
2023-06-30 2023-08-24  5500524000000.0
2023-09-30 2023-10-25  5516388000000.0
2023-10-01 2023-10-25  5516388000000.0
----------
               信息发布日期             资产总计
2023-06-30 2023-04-25  5455897000000.0
2023-09-30 2023-08-24  5500524000000.0
2023-10-01 2023-08-24  5500524000000.0


### 多重映射

数据库数据示例：

![Wide_Table_MultiMapping_Demo](../images/Wide_Table_MultiMapping_Demo.png)

In [8]:
# 对于一个报告期，可能有多份财报（包括修正），因此 MultiMapping=True 时一个时点下对应着多个值
DTs = [dt.datetime(2023, 9, 30), dt.datetime(2023, 12, 31)]
IDs = ["000001.SZ"]

FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": 0,
    "DTField": "截止日期",
    "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1"},
    "MultiMapping": True
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

# 可以通过指定算子，将多个值合并成一个值
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": 0,
    "DTField": "截止日期",
    "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1"},
    "MultiMapping": True,
    "Operator": lambda x: x.mean(),
    "OperatorDataType": "double"
})
Data = FT.readData(factor_names=["资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

                                                       信息发布日期  \
2023-09-30                              [2023-10-25 00:00:00]   
2023-12-31  [2024-03-15 00:00:00, 2024-04-20 00:00:00, 202...   

                                                         资产总计  
2023-09-30                               [5516388000000.0000]  
2023-12-31  [5587116000000.0000, 5587116000000.0000, 55871...  
                    资产总计
2023-09-30  5.516388e+12
2023-12-31  5.587116e+12


## FeatureTable

继承自 WideTable

数据库数据示例：

![Feature_Table_Demo](../images/Feature_Table_Demo.png)

In [3]:
FT = FDB.getTable("A股证券主表")
print(qs_help(FT))

类型: _FeatureTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: A股证券主表
QSID: 763d905c878358fb0bc484774af32baca6547c02f820a2e9301998c9a255b5c6
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'A股证券主表'
    * FilterCondition(筛选条件): <class 'str'>, 默认值 '', 形成 SQL 查询时附加到 WHERE 子句上的条件. 比如 "({Table}.field1>10) AND ({Table}.field2 IN ('a','b')", 其中 {Table} 会自动替换为相应的数据库表名, 当前取值: ''
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'FeatureTable'
    * PreFilterID(预筛选ID): <class 'bool'>, 默认值 True, 是否在 SQL 查询中筛选 ID, 如果为 True, 则在形成的 SQL 查询中的 WHERE 子句中会有 {Table}.ID字段 IN (...) 条件, 否则为 {Table}.ID字段 IS NOT NULL. 如果提取数据的 ID 不多，建议为 True, 当前取值: True
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: None
    * IDField(ID字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示 ID 维度的字段名, 当前取值: None
    * DTFmt(时点格式): <class 'str'>, 默认值 '', 当前取值: ''
    * DateFmt(日期格式): <class 'str'>, 默认值

In [ ]:
# FeatureTable 每个时点的数据都一样
DTs = [dt.datetime(2025, 1, 1), dt.datetime(2025, 1, 2)]
IDs = ["000001.SZ"]

FT = FDB.getTable("A股证券主表", args={})
Data = FT.readData(factor_names=["证券简称", "上市日期"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

            证券简称       上市日期
2025-01-01  平安银行 1991-04-03
2025-01-02  平安银行 1991-04-03


## TimeSeriesTable

数据库数据示例：

![TimeSeries_Table_Demo](../images/TimeSeries_Table_Demo.png)

## NarrowTable

数据库数据示例：

![Narrow_Table_Demo](../images/Narrow_Table_Demo.png)

## MappingTable

数据库数据示例：

![Mapping_Table_Demo](../images/Mapping_Table_Demo.png)

In [6]:
FT = FDB.getTable("公司行业划分表")
print(qs_help(FT))

类型: _MappingTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 公司行业划分表
QSID: 813dcf05c55edb4a74beb91efcbbc558c2023fea130d99df9d8da4e38907bda8
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '公司行业划分表'
    * FilterCondition(筛选条件): <class 'str'>, 默认值 '', 形成 SQL 查询时附加到 WHERE 子句上的条件. 比如 "({Table}.field1>10) AND ({Table}.field2 IN ('a','b')", 其中 {Table} 会自动替换为相应的数据库表名, 当前取值: ''
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'MappingTable'
    * PreFilterID(预筛选ID): <class 'bool'>, 默认值 True, 是否在 SQL 查询中筛选 ID, 如果为 True, 则在形成的 SQL 查询中的 WHERE 子句中会有 {Table}.ID字段 IN (...) 条件, 否则为 {Table}.ID字段 IS NOT NULL. 如果提取数据的 ID 不多，建议为 True, 当前取值: True
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '信息发布日期'
    * IDField(ID字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示 ID 维度的字段名, 当前取值: None
    * DTFmt(时点格式): <class 'str'>, 默认值 '', 当前取值: ''
    * DateFmt(日期格式): <class 'str'

In [ ]:
# MappingTable 位于 DTField 和 EndDTField 之间的时点都会填充一样的值
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
IDs = ["600136.SH"]

FT = FDB.getTable("公司行业划分表", args={"AdditionalCondition": {"行业划分标准": "37"}})
Data = FT.readData(factor_names=["一级行业代码", "一级行业名称"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

           一级行业代码 一级行业名称
2018-02-01     22   基础化工
2019-12-03     63     传媒


## ConstituentTable

数据库数据示例：

![Constituent_Table_Demo](../images/Constituent_Table_Demo.png)

In [10]:
FT = FDB.getTable("指数成份")
print(qs_help(FT))

类型: _ConstituentTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 指数成份
QSID: 1da8927a2b34f7d33df87ead4de7902cbc3e7ae8868197d1b688b9cbdd82251f
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '指数成份'
    * FilterCondition(筛选条件): <class 'str'>, 默认值 '', 形成 SQL 查询时附加到 WHERE 子句上的条件. 比如 "({Table}.field1>10) AND ({Table}.field2 IN ('a','b')", 其中 {Table} 会自动替换为相应的数据库表名, 当前取值: ''
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'ConstituentTable'
    * PreFilterID(预筛选ID): <class 'bool'>, 默认值 True, 是否在 SQL 查询中筛选 ID, 如果为 True, 则在形成的 SQL 查询中的 WHERE 子句中会有 {Table}.ID字段 IN (...) 条件, 否则为 {Table}.ID字段 IS NOT NULL. 如果提取数据的 ID 不多，建议为 True, 当前取值: True
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '入选日期'
    * IDField(ID字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示 ID 维度的字段名, 当前取值: None
    * DTFmt(时点格式): <class 'str'>, 默认值 '', 当前取值: ''
    * DateFmt(日期格式): <class 'str'

In [11]:
# ConstituentTable 位于 DTField 和 EndDTField 之间的时点都会填充 1，其余时点为 0 或者 nan
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
IDs = ["000001.SZ"]

FT = FDB.getTable("指数成份")
Data = FT.readData(factor_names=["3145", "46"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

           3145 46
2018-02-01    1  0
2019-12-03    1  0


## FinancialTable

数据库数据示例：

![Financial_Table_Demo](../images/Financial_Table_Demo.png)

In [3]:
FT = FDB.getTable("资产负债表_新会计准则")
print(qs_help(FT))

类型: _FinancialTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 资产负债表_新会计准则
QSID: 667c748c3dd910d8df37ff97a35edd3d5950bdfa42a70dada8a111f8048062fb
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '资产负债表_新会计准则'
    * Parallel(并行计算): <class 'bool'>, 默认值 True, 当前取值: True
    * TaskExecutor(并行执行器): typing.Optional[concurrent.futures._base.Executor], 默认值 None, 给到节点用于并行计算, 当前取值: None
    * FilterCondition(筛选条件): <class 'str'>, 默认值 '', 形成 SQL 查询时附加到 WHERE 子句上的条件. 比如 "({Table}.field1>10) AND ({Table}.field2 IN ('a','b')", 其中 {Table} 会自动替换为相应的数据库表名, 当前取值: ''
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'FinancialTable'
    * PreFilterID(预筛选ID): <class 'bool'>, 默认值 True, 是否在 SQL 查询中筛选 ID, 如果为 True, 则在形成的 SQL 查询中的 WHERE 子句中会有 {Table}.ID字段 IN (...) 条件, 否则为 {Table}.ID字段 IS NOT NULL. 如果提取数据的 ID 不多，建议为 True, 当前取值: True
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '截止日期'
    *

In [14]:
# FinancialTable
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
IDs = ["000001.SZ"]

FT = FDB.getTable("资产负债表_新会计准则", args={
    "ReportDate": "年报",
    "CalcType": "最新"
})
Data = FT.readData(factor_names=["资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

                    资产总计
2018-02-01  2.953434e+12
2019-12-03  3.418592e+12
